# AE2: manejo de archivos CSV, JSON y TXT

Flujo de trabajo:

```
Descarga manual → importar → modificar → guardar nuevo CSV y JSON
                → tercera clase → resumen TXT con URL de origen
```

El notebook solo coordina: importa las tres clases, crea los objetos y llama a sus métodos. Toda la lógica está en `manejador_csv.py`, `manejador_json.py` y `generador_resumen_txt.py`.

Antes de ejecutar: descarga manualmente un CSV y un JSON de Hugging Face, guárdalos en `datos/` y escribe sus URL en la primera celda de código.

## 1. Origen de los datos y rutas

Las URL son las **URL específicas** de cada archivo en Hugging Face (la del archivo, no la de la página del dataset). Se guardan en variables para usar las mismas en el resumen. Las rutas son relativas a la carpeta `AE2`.

In [1]:
# TODO: Url de Hugging Face
URL_CSV = "https://huggingface.co/datasets/ytRankAI/Top_100_YouTube_Channels/blob/main/top_100_youtube_channels.csv"
URL_JSON = "https://huggingface.co/datasets/pyrihtm/movies_100/blob/main/movies_100.json"

# Archivos originales (descargados a mano).
RUTA_CSV_ORIGINAL = "datos/top_100_youtube_channels.csv"
RUTA_JSON_ORIGINAL = "datos/movies_100.json"

# Archivos nuevos: se guardan con otro nombre para conservar los originales.
RUTA_CSV_MODIFICADO = "datos/top_100_youtube_channels_datos_modificado.csv"
RUTA_JSON_MODIFICADO = "datos/movies_100_datos_modificado.json"

RUTA_RESUMEN = "resumen.txt"

## 2. Importar las tres clases

El notebook busca la carpeta `AE2` hacia arriba desde donde se ejecute y trabaja desde ahí, así las rutas relativas siempre funcionan.

In [2]:
import os
import sys
from pathlib import Path

AE2_DIR = Path.cwd().resolve()
while AE2_DIR != AE2_DIR.parent and not (AE2_DIR / "manejador_csv.py").exists():
    if (AE2_DIR / "AE2" / "manejador_csv.py").exists():
        AE2_DIR = AE2_DIR / "AE2"
        break
    AE2_DIR = AE2_DIR.parent

os.chdir(AE2_DIR)
sys.path.insert(0, str(AE2_DIR))

from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON
from generador_resumen_txt import GeneradorResumenTxt

print(f"Carpeta de trabajo: {Path.cwd()}")

Carpeta de trabajo: /workspaces/Big-Data/AE2


## 3. Inicializar los objetos de las clases CSV y JSON

In [3]:
manejador_csv = ManejadorCSV()
manejador_json = ManejadorJSON()

## 4. Importar los archivos locales

`importar_csv()` usa `csv.reader` y `importar_json()` usa `json.load`.

In [4]:
manejador_csv.importar_csv(RUTA_CSV_ORIGINAL)
manejador_json.importar_json(RUTA_JSON_ORIGINAL)

print(f"CSV importado:  {manejador_csv.registros_importados} registros")
print(f"JSON importado: {manejador_json.registros_importados} registros")

CSV importado:  100 registros
JSON importado: 100 registros


## 5. Modificar el CSV

Tres cambios hechos con métodos de la clase `ManejadorCSV`: actualizar un valor, eliminar un registro y agregar uno nuevo. Se muestra el antes y el después.



In [5]:
COLUMNA_CSV = manejador_csv.encabezado[-1]  # TODO: opcional, elige otra columna

manejador_csv.mostrar("ANTES", cantidad=3)
print()

manejador_csv.actualizar_valor(0, COLUMNA_CSV, "VALOR_ACTUALIZADO")
manejador_csv.eliminar_registro(1)

nuevo_csv = dict(zip(manejador_csv.encabezado, manejador_csv.registros[0]))
nuevo_csv[manejador_csv.encabezado[0]] = "registro_nuevo"
manejador_csv.agregar_registro(nuevo_csv)

manejador_csv.mostrar("DESPUÉS", cantidad=3)
print("...")
print(f"  [{len(manejador_csv.registros) - 1}] {manejador_csv.registros[-1]}  <- registro agregado")

ANTES
Columnas (10): ['channel_id', 'title', 'canonical_channel_url', 'subscriber_count', 'video_count', 'view_count', 'category', 'country', 'channel_image', 'channel_url']
Registros: 100
  [0] ['UCX6OQ3DkcsbYNE6H8uQQuVA', 'MrBeast', 'http://www.youtube.com/@MrBeast', '319000000', '820', '60516236462', 'People & Blogs', 'United States', 'https://yt3.googleusercontent.com/fxGKYucJAVme-Yz4fsdCroCFCrANWqw0ql4GYuvx8Uq4l_euNJHgE-w9MTkLQA805vWCi-kE0g=s900-c-k-c0x00ffffff-no-rj', 'https://www.ytrank.com/channel/mrbeast-UCX6OQ3DkcsbYNE6H8uQQuVA']
  [1] ['UCq-Fj5jknLsUf-MWSy4_brA', 'T-Series', 'http://www.youtube.com/@tseries', '275000000', '21831', '269122247099', 'Music', 'India', 'https://yt3.googleusercontent.com/XE7Iq8jvJ07ptMc-HxZR_V-2XgXCb0i06i4E_dypl7xSR655WXaQeglfqNuEeuwH3oM9RKVodQ=s900-c-k-c0x00ffffff-no-rj', 'https://www.ytrank.com/channel/t-series-UCq-Fj5jknLsUf-MWSy4_brA']
  [2] ['UCbCmjCuTUZos6Inko4u57UQ', 'Cocomelon - Nursery Rhymes', 'http://www.youtube.com/@CoComelon', '183000

## 6. Modificar el JSON

Los mismos tres cambios con los métodos de `ManejadorJSON`.

In [6]:
CAMPO_JSON = manejador_json.campos()[-1]  # TODO: opcional, elige otro campo

manejador_json.mostrar("ANTES", cantidad=2)
print()

manejador_json.actualizar_valor(0, CAMPO_JSON, "VALOR_ACTUALIZADO")
manejador_json.eliminar_registro(1)

nuevo_json = dict(manejador_json.obtener_registros()[0])
nuevo_json[manejador_json.campos()[0]] = "registro_nuevo"
manejador_json.agregar_registro(nuevo_json)

manejador_json.mostrar("DESPUÉS", cantidad=2)
print("...")
ultimo = manejador_json.obtener_registros()[-1]
print(f"  [{len(manejador_json.obtener_registros()) - 1}] {ultimo}  <- registro agregado")

ANTES
Registros: 100
  [0] {"instruction": "Return schema.org Entities and their Properties in JSON-LD format as an array strictly based on the text below. Don't add any common knowledge that is not stated in this text.", "input": " \"Big Naturals\" Julies Jewels (TV Episode 2014) - IMDb Big Naturals S14 . E38 All episodes All Cast & crew IMDbPro All topics Julies Jewels Episode aired Sep 24, 2014 YOUR RATING Rate Adult Add a plot in your language Stars Julie Kay Mirko Steel See production info at IMDbPro YOUR RATING Rate Stars Julie Kay Mirko Steel See production info at IMDbPro See production info at IMDbPro Top cast Edit Julie Kay Mirko Steel All cast & crew Production, box office & more at IMDbPro Storyline Edit User reviews Be the first to review Be the first to review Top picks Sign in to rate and Watchlist for personalized recommendations Sign in Details Edit Release date September 24, 2014 (United States) See more company credits at IMDbPro Technical specs Edit Color Color Soun

## 7. Guardar el nuevo CSV y el nuevo JSON

`guardar_csv()` usa `csv.writer` (escribe primero el encabezado) y `guardar_json()` usa `json.dump`. Después se leen de nuevo los archivos guardados con objetos nuevos para verificar que contienen los cambios.

In [7]:
ruta_csv_nueva = manejador_csv.guardar_csv(RUTA_CSV_MODIFICADO)
ruta_json_nueva = manejador_json.guardar_json(RUTA_JSON_MODIFICADO)
print(f"CSV guardado en:  {ruta_csv_nueva}")
print(f"JSON guardado en: {ruta_json_nueva}")

# Verificación: se vuelven a leer los archivos guardados
verif_csv = ManejadorCSV()
verif_csv.importar_csv(ruta_csv_nueva)
verif_json = ManejadorJSON()
verif_json.importar_json(ruta_json_nueva)

assert verif_csv.encabezado == manejador_csv.encabezado, "El encabezado del CSV cambió"
assert verif_csv.registros == manejador_csv.registros, "El CSV guardado no coincide"
assert verif_csv.registros[0][verif_csv.encabezado.index(COLUMNA_CSV)] == "VALOR_ACTUALIZADO"
assert verif_json.obtener_registros() == manejador_json.obtener_registros(), "El JSON guardado no coincide"
assert verif_json.obtener_registros()[0][CAMPO_JSON] == "VALOR_ACTUALIZADO"

print(f"\nCSV verificado:  {verif_csv.registros_importados} registros "
      f"(original: {manejador_csv.registros_importados})")
print(f"JSON verificado: {verif_json.registros_importados} registros "
      f"(original: {manejador_json.registros_importados})")

CSV guardado en:  datos/top_100_youtube_channels_datos_modificado.csv
JSON guardado en: datos/movies_100_datos_modificado.json

CSV verificado:  100 registros (original: 100)
JSON verificado: 100 registros (original: 100)


## 8. Generar el resumen .txt

`GeneradorResumenTxt` recibe las mismas URL que se definieron al inicio y los resultados reales de los dos manejadores (`resultado()`).

In [8]:
generador = GeneradorResumenTxt(url_csv=URL_CSV, url_json=URL_JSON)
ruta_resumen = generador.generar_resumen(
    manejador_csv.resultado(),
    manejador_json.resultado(),
    RUTA_RESUMEN,
)

print(f"Resumen guardado en: {ruta_resumen}\n")
print(generador.leer_resumen(ruta_resumen))

Resumen guardado en: resumen.txt

RESUMEN DE LA ACTIVIDAD AE2: MANEJO DE ARCHIVOS CSV, JSON Y TXT
Fecha de generación: 2026-10-29 03:28:15

1. URL de origen
- CSV:  https://huggingface.co/datasets/ytRankAI/Top_100_YouTube_Channels/blob/main/top_100_youtube_channels.csv
- JSON: https://huggingface.co/datasets/pyrihtm/movies_100/blob/main/movies_100.json

2. Archivo CSV
- Archivo importado: datos/top_100_youtube_channels.csv
- Archivo nuevo guardado: datos/top_100_youtube_channels_datos_modificado.csv
- Modificaciones realizadas:
    * Actualizado el registro 0, columna 'channel_url': 'https://www.ytrank.com/channel/mrbeast-UCX6OQ3DkcsbYNE6H8uQQuVA' -> 'VALOR_ACTUALIZADO'
    * Eliminado el registro 1
    * Agregado un registro nuevo en la posición 99

3. Archivo JSON
- Archivo importado: datos/movies_100.json
- Archivo nuevo guardado: datos/movies_100_datos_modificado.json
- Modificaciones realizadas:
    * Actualizado el registro 0, campo 'output': {'@context': 'https://schema.org', '@